# 03 · Federated learning on IID hospitals

The first cost to measure is the one no hospital can avoid: **decentralization
itself**. Here the training set is dealt out at random, so every simulated
hospital holds the same label mix and the same devices. Any gap to the
centralized baseline is caused only by training in pieces.

Prerequisite:

```bash
for n in 5 10 20; do uv run python scripts/train_federated.py --config fedavg_iid_$n.yaml; done
```

## FedAvg in one paragraph

Each round the server sends the current global model to every hospital. Each
hospital trains it for one epoch on its own records and sends the weights
back. The server averages them, weighting each hospital by its number of
records, and that average is the next global model. No ECG leaves a hospital;
what travels is the weights, about 6.3 MB per copy for this network (1.6M
parameters).

With every hospital training one epoch per round, a round is one pass over the
whole training set, so 50 rounds cost the same compute as 50 centralized
epochs. Models are selected the same way in both settings: at the round with
the best validation AUROC, scored on the server's unsplit validation fold.

The client (`fedecg.federated.client`) is a Flower `NumPyClient`; the averaging
is Flower's own `FedAvg` strategy. The clients run one after another in this
process instead of in Ray workers (`fedecg.federated.simulation` explains why).

In [ ]:
%matplotlib inline
import pandas as pd

from fedecg.paths import TABLES_DIR
from fedecg.viz import plot_curves

experiments = pd.read_csv(TABLES_DIR / "experiments.csv")
baseline = experiments.set_index("run").loc["centralized"]


def history(run: str) -> pd.DataFrame:
    return pd.read_csv(TABLES_DIR / f"{run}_history.csv")


def with_gap(rows: pd.DataFrame) -> pd.DataFrame:
    """Add the AUROC gap to the centralized baseline, the number each phase is about."""
    rows = rows.assign(gap=rows["macro_auroc"] - baseline["macro_auroc"])
    return rows.set_index("setting")

## The cost of decentralization

In [ ]:
iid = experiments[(experiments["phase"] == 4) & (experiments["partition"] == "iid")]
columns = ["n_clients", "macro_auroc", "gap", "best_step", "communication_mb", "seconds"]
with_gap(iid.sort_values("n_clients"))[columns].round(4)

In [ ]:
curves = {"centralized (per epoch)": history("centralized")}
curves |= {f"FedAvg, {row.n_clients} hospitals": history(row.run) for row in iid.itertuples()}
fig = plot_curves(curves, step_label="epoch (centralized) or round (federated)")

## What this means

- **Decentralization has a price even with identical hospitals, and it grows
  with their number:** −0.007 AUROC at 5 hospitals, −0.016 at 10, −0.026 at
  20. All three are well above the ~0.002 seed noise, and all three are within
  0.002 of what the smaller network lost before the baseline was tuned.
- **Part of the price is speed, not a ceiling.** The federated curves are still
  rising at round 50, while the centralized one peaked at epoch 33. With the
  compute held equal, FedAvg makes less progress per pass over the data: each
  hospital takes a few local steps from the global model, and averaging
  partially cancels them. More rounds would narrow the gap, at a
  proportionally higher communication cost.
- **Communication scales with the number of hospitals and the model.** 50
  rounds cost 3.1 GB of weight traffic at 5 hospitals and 12.6 GB at 20 for
  this 1.6M-parameter network, four times what the 32-channel version needed,
  against zero for the centralized model.